In [4]:
# ── Cell 1: Reproducibility Header ────────────────────────────────────────────
# Every notebook in IIT414W starts here. Do not skip this block.

import sys, random
import numpy as np
import warnings
import platform

RANDOM_SEED = 414  # Course constant. Do not change.
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
warnings.filterwarnings('ignore', category=FutureWarning)

# Environment check
print(f'Python  : {sys.version.split()[0]}')
print(f'NumPy   : {np.__version__}')
print(f'Seed    : {RANDOM_SEED}')
print("\nSystem: " + platform.system())
print("Distro: " + platform.release())

Python  : 3.14.3
NumPy   : 2.2.3
Seed    : 414

System: Linux
Distro: 6.19.6-2-cachyos


In [5]:
# ── Cell 2: Dependency Guard ───────────────────────────────────────────────
# Ensures all required packages are installed in the active kernel.
# Safe to re-run: pip will skip already-installed packages.

import importlib, subprocess, sys

_REQUIRED = {
    'numpy': 'numpy',
    'pandas': 'pandas',
    'sklearn': 'scikit-learn',
    'matplotlib': 'matplotlib',
    'seaborn': 'seaborn',
    'fastf1': 'fastf1',
}

_missing = []
for _mod, _pip in _REQUIRED.items():
    try:
        importlib.import_module(_mod)
    except ModuleNotFoundError:
        _missing.append(_pip)

if _missing:
    print(f'Installing missing packages: {_missing}')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet'] + _missing)
    print('Done. Packages installed successfully.')
else:
    print('All required packages already installed ✓')

# ── Library Imports ───────────────────────────────────────────────
import os                        # Working directory checks
import subprocess                # Git command checks
import importlib                 # Runtime dependency checks
import numpy as np               # Numeric support
import pandas as pd              # Tables and diagnostics
import matplotlib.pyplot as plt  # Plotting
import seaborn as sns            # Statistical plotting
import fastf1                    # Formula 1 data access

print(f'fastf1  : {fastf1.__version__}')
print(f'pandas  : {pd.__version__}')

All required packages already installed ✓
fastf1  : 3.8.1
pandas  : 2.2.3


In [6]:
# ── Cell 2.1: Cache and validations ───────────────────────────────────────────────

required = ['numpy', 'pandas', 'sklearn', 'matplotlib', 'seaborn', 'fastf1']
rows = []
for pkg in required:
    mod = importlib.import_module(pkg)
    rows.append((pkg, getattr(mod, '__version__', 'n/a')))

print(pd.DataFrame(rows, columns=['package', 'version']).to_string(index=False))
print(f'Working directory: {os.getcwd()}')

# FastF1 cache — auto-create if missing
cache_path = os.path.join(os.path.dirname(os.getcwd()), '..', 'data', 'fastf1_cache')
cache_path = os.path.abspath(cache_path)
os.makedirs(cache_path, exist_ok=True)
fastf1.Cache.enable_cache(cache_path)
print(f'FastF1 cache enabled: {cache_path}')

print(subprocess.run(['git', '--version'], capture_output=True, text=True, check=False).stdout.strip())
log = subprocess.run(['git', 'log', '--oneline', '-5'], capture_output=True, text=True, check=False)
print('Recent commits:')
print(log.stdout.strip() if log.returncode == 0 else 'No commit history available in this folder.')

   package version
     numpy   2.2.3
    pandas   2.2.3
   sklearn   1.6.1
matplotlib  3.10.0
   seaborn  0.13.2
    fastf1   3.8.1
Working directory: /home/inzenfenix/Documents/GitHub/iit414w-lab01-STS17
FastF1 cache enabled: /home/inzenfenix/Documents/data/fastf1_cache
git version 2.53.0
Recent commits:
c5ff40c Import all libreries & declaration of the development environment
14e5abd Files
2ff2fbb Initial commit


## 1 Domain Heuristic Baseline

In [ ]:
# ── Cell 3: Domain Heruistic Baseline ───────────────────────────────────────────────
# Check the rule that if you start at more than the 10th position on the grid, you don't end up on the top 10.

import fastf1
import pandas as pd


def predict_bottom_pos_heuristic(grid_position):
    return 1 if grid_position > 10 else 0

print("------------")
print("\nRule: If you start at the bottom of the grid, you finish at the end.\n")
print("------------")

validation_results = []
race_nums = [22, 23, 24] # Las Vegas, Qatar, Abu Dhabi

for race_num in race_nums:
    session = fastf1.get_session(2024, race_num, 'R')
    session.load(laps=False, telemetry=False)
    results = session.results[['FullName', 'GridPosition', 'ClassifiedPosition']]
    
    for _, row in results.iterrows():
        # Handle DNFs or unclassified if necessary
        # Also get the data as 0s and 1s to calculate accuracy
        binary_converter = lambda x: 1 if x > 10 else 0
        try:
            actual_pos = int(row['ClassifiedPosition'])
            actual_last = binary_converter(actual_pos)
            predicted_last = predict_bottom_pos_heuristic(row['GridPosition'])
            
            validation_results.append({
                'actual': actual_last,
                'predicted': predicted_last
            })
        except ValueError:
            continue # Skip non-classified drivers

df_val = pd.DataFrame(validation_results)

# 4.2 GET Accuracy
accuracy = (df_val['actual'] == df_val['predicted']).sum() / len(df_val)

print("\n\n---- 4.2 ACCURACY ----")
print(f"Heuristic Baseline Accuracy: {accuracy:.2%}")


core           INFO 	Loading data for Las Vegas Grand Prix - Race [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['63', '44', '55', '16', '1', '4', '81', '27', '22', '11', '14', '20', '24', '43', '18', '30', '31', '77', '23', '10']
core           INFO 	Loading data for Qatar Grand Prix - Race [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '16', '81', '63', '10', '55', '14', '24', '20', '4', '77', '44', '22', '30', '23', '27', '11', '18', '43', '31']
core           INFO 	Loading data 

Rule: If you start at the bottom of the grid, you finish at the end.


---- 4.2 ACCURACY ----
Heuristic Baseline Accuracy: 88.00%


### 4.3 Reflection

> Is this accuracy good enough to make decisions with? 

- It's enough to make a starting point, to notice there's something more to the obtained data that meets the eye, if dug deeper you could find more information, you could build a better model and get a better understanding of the presented information

> What could accuracy be hiding?

- What part of the data is actually real, meaning if I were to put this into an actual model or even with this heuristic rule, we don't actually know how much of the data is either false positive or false negative, or true positive or true negative, we just know how much of the data was accurate to our rule.

> what would be a baseline that always predicts "bottom" score?

- I don't think there's a way to always predict the top 10 score or bottom score, for the same reason why ChatGPT or any LLM model gets like, 30-90% of a question wrong, this systems are based on rules whose baseline is always a possibility, no just 1 or 0, but rather a number in between, besides, in a race there are so many unpredictable variables that It would be near impossible to get a complete prediction of anything, for example, imagine that there's an earthquake, you can only predict something like that only seconds before It happens with current technology, so a baseline that always predicts anything is impossible to achieve.